# ShellWhisperer-1.5B Unsloth QLoRA Training
Fine-tune Qwen2.5-Coder-1.5B-Instruct for shell/bash command generation

**Hardware:** T4 GPU (free Colab)
**Method:** 4-bit QLoRA with Unsloth
**LoRA rank:** 16, **Epochs:** 5, **Target modules:** 7

In [ ]:
#@title 1. Install Unsloth
%%capture
!pip install unsloth
!pip install --upgrade torch transformers trl peft accelerate bitsandbytes
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
#@title 2. Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 3. Load Model with Unsloth (4-bit quantization)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

In [ ]:
#@title 4. Apply LoRA adapters (r=16, 7 target modules)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#@title 5. Download training data from GitHub
import json
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/KingLabsA/fableforge-training/main/"

def download_jsonl(filename):
    url = BASE_URL + filename
    print(f"Downloading {url}...")
    urllib.request.urlretrieve(url, filename)
    with open(filename) as f:
        data = [json.loads(line) for line in f if line.strip()]
    print(f"  Loaded {len(data)} examples")
    return data

train_data = download_jsonl("shellwhisperer_train.jsonl")
val_data = download_jsonl("shellwhisperer_val.jsonl")
print(f"\nTotal train: {len(train_data)}, Total val: {len(val_data)}")

In [ ]:
#@title 6. Format data into ChatML
def format_to_chatml(example):
    """Convert Alpaca or ChatML format to standardized ChatML for Qwen."""
    if "messages" in example:
        messages = example["messages"]
    elif "instruction" in example:
        messages = []
        if example.get("system"):
            messages.append({"role": "system", "content": example["system"]})
        messages.append({"role": "user", "content": example["instruction"]})
        messages.append({"role": "assistant", "content": example["output"]})
    else:
        raise ValueError(f"Unknown format: {list(example.keys())}")
    return {"messages": messages}

train_formatted = [format_to_chatml(ex) for ex in train_data]
val_formatted = [format_to_chatml(ex) for ex in val_data]

print(f"Example formatted message:")
print(json.dumps(train_formatted[0], indent=2)[:500])

In [ ]:
#@title 7. Configure SFT Trainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    dataset_text_field="messages",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/kaggle/working/shellwhisperer-output",
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)
print("Trainer configured!")
print(f"Effective batch size: {4 * 4}")
print(f"Total steps: ~{len(train_formatted) * 5 // (4 * 4)}")

In [ ]:
#@title 8. Train!
import time
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

In [ ]:
#@title 9. Evaluate
eval_results = trainer.evaluate()
print(f"Eval loss: {eval_results['eval_loss']:.4f}")

In [ ]:
#@title 10. Save LoRA adapters
model.save_pretrained("/content/shellwhisperer-1.5b-lora")
tokenizer.save_pretrained("/content/shellwhisperer-1.5b-lora")
print(f"LoRA adapters saved to /content/shellwhisperer-1.5b-lora")

# List saved files
import os
for f in os.listdir("/content/shellwhisperer-1.5b-lora"):
    path = os.path.join("/content/shellwhisperer-1.5b-lora", f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

In [ ]:
#@title 11. Merge LoRA into base model and save (full model)
model.save_pretrained_merged("/content/shellwhisperer-1.5b-merged", tokenizer)
print("Merged model saved to /content/shellwhisperer-1.5b-merged")
print(f"Total size: {sum(os.path.getsize(os.path.join('/content/shellwhisperer-1.5b-merged', f)) for f in os.listdir('/content/shellwhisperer-1.5b-merged')) / 1e9:.2f} GB")

In [ ]:
#@title 12. Quick inference test
FastLanguageModel.for_inference(model)

test_prompts = [
    "List all Python files in the current directory recursively",
    "Find all files larger than 100MB",
    "Kill a process running on port 8080",
    "Count the number of lines in all .js files",
    "Show disk usage sorted by size",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are ShellWhisperer, an expert at generating accurate shell/bash commands. Provide ONLY the command(s), no explanation."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {response.strip()}")

In [ ]:
#@title 13. Upload to HuggingFace
# First, set your HuggingFace token as a secret in Colab:
# Click the key icon in the left sidebar > Add secret > Name: HF_TOKEN > Value: your_token
#
# Alternatively, use huggingface-cli login

from huggingface_hub import HfApi
import getpass

# Option 1: Use Colab secrets
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except:
    # Option 2: Prompt for token
    hf_token = getpass.getpass('Enter your HuggingFace token: ')

api = HfApi(token=hf_token)

# Upload merged model
print("Uploading ShellWhisperer-1.5B to HuggingFace...")
api.upload_folder(
    folder_path="/content/shellwhisperer-1.5b-merged",
    repo_id="fableforge-ai/ShellWhisperer-1.5B",
    repo_type="model",
    commit_message="Upload ShellWhisperer-1.5B fine-tuned with Unsloth QLoRA"
)
print("Upload complete!")
print(f"Model available at: https://huggingface.co/fableforge-ai/ShellWhisperer-1.5B")